# Graph Clustering

## Imports

In [ ]:
# !pip install cdlib
# !pip install igraph
# !pip install leidenalg

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from matplotlib import cm
import json
from cdlib import algorithms
from graphUtilities import *
import json

# Audio
from src.ftm import rectangular_drum
from IPython.display import Audio, display
from scipy.io.wavfile import write

constants = {
    "x1": 0.4,
    "x2": 0.4,
    "h": 0.03,
    "l0": np.pi,
    "m1": 10,
    "m2": 10,
    "sr": 22050,
    "dur":2**16
}

results_path = "results\\Clustering"

##  Community detection technics
- Spectral clustering
    - https://scikit-learn.org/stable/modules/generated/sklearn.cluster.SpectralClustering.html
- Cluster detection by girvan newman algorithm 
    - https://en.wikipedia.org/wiki/Girvan%E2%80%93Newman_algorithm
- Louvain community detection
- Leiden
- walktrap 

## Calculated score for clustering method

The score of each iteration is calculated by the modularity index :

$Q = \frac{1}{2m} \sum_{ij} \left( A_{ij} - \gamma\frac{k_ik_j}{2m}\right)
    \delta(c_i,c_j)$
    
where 
 $m$ is the number of edges, 
 
 $A$ is the adjacency matrix of G, 
 
 $k_i$is the (weighted) degree of $i$
 
 $\gamma$ is the resolution parameter (1 by default)
 
 $\delta (c_i,c_j)$ is 1 if $i$ and $j$ are in the same community else 0.

## Laod graphs

In [ ]:
g_list = []
graph_path_list = []
edge_type = "invDist" # "invDist"
k_bounds = (10,48)
nb_hubs = 500

### Compute clustering with Louvain, Leiden and walktrap method

In [ ]:
nb_connected_component = []
clustering_result = {}
clustering_result_init(clustering_result)

for k in range(k_bounds[0],k_bounds[-1]+1):
    
    #Load the graph
    graph_Name = "KnnG_Nhubs" + str(nb_hubs) +  "_K" + str(k) +".graphml"
    print('\n' + "------------------- " + graph_Name + " -------------------" + '\n')
    g = load_graph(graph_Name,edge_type='dist',verbose=True)
    
    #Graph connected component
    components_size = connectedComponentsHisto(g,graph_Name,plot=True)
    nb_connected_component.append(len(components_size))
    #Plot histrogram of degree
    plot_degree_histo(g,graph_Name)
    
    #Perform clustering
    testGraphClustering(g,clustering_result,verbose=True)
print("Done !")

In [ ]:
#Save the results

#clustering_result
json_str = json.dumps(clustering_result, indent=4)
with open(results_path + "\\clustering_result.json", "w") as f:
    f.write(json_str)
print(results_path + "clustering_result.json")

##Number of connected components       
json_str = json.dumps(nb_connected_component, indent=4)
with open(results_path + "\\nb_connected_component.json", "w") as f:
    f.write(json_str)

### Print scores

In [ ]:
# Load data from JSON files
with open(results_path + "\\clustering_result.json" ) as json_file:
    clustering_result = json.load(json_file)

with open(results_path + "\\nb_connected_component.json" ) as json_file:
    nb_connected_component = json.load(json_file)

#Find first graph with a signle connected component
f_idx = firstSigneComponent(nb_connected_component)+k_bounds[0]

# scoring
plotScoring(clustering_result,k_bounds,f_idx)

#Number of connected components
plotComponentsCurve(nb_connected_component,k_bounds,idxSingleComponent=f_idx)

#Number of clusters
plotNbClusters(clustering_result,k_bounds,idxSingleComponent=f_idx)

# Visualize communities

In [ ]:
#Load the graph
k = 35
graph_Name = "KnnG_Nhubs" + str(nb_hubs) +  "_K" + str(k) +".graphml"
print('\n' + "------------------- " + graph_Name + " -------------------" + '\n')
g = load_graph(graph_Name,edge_type='dist',verbose=True)


In [ ]:
method = 'louvain'
for community_index in range(len(clustering_result["leiden"][k])):
    community_index = 0
    community = clustering_result["leiden"][k][community_index]
    print(len(community))
    G = g.subgraph(community)
    plt.figure()
    nx.draw(G)
    plt.show()
    connectedComponentsHisto(G, "leiden com : k = "+ str(k) + "com = " +  str(community_index), plot=True)

### Listen to samples of communities

In [ ]:
#Load the graph
k = 35
graph_Name = "KnnG_Nhubs" + str(nb_hubs) +  "_K" + str(k) +".graphml"
print('\n' + "------------------- " + graph_Name + " -------------------" + '\n')
g = load_graph(graph_Name,edge_type='dist',verbose=True)

communities = clustering_result["leiden"][k]
for c in range(len(communities)):
    # create a dict of degree
    d_a = {}
    for node in communities[c]:
        d_a[node] = g.degree()[node]
        
    #find node whith the highest degree
    node_key = max(d_a, key=lambda k: d_a[k])
    idx = 0
    listNode = list(d_a.keys())
    while listNode[idx] != node_key:
        idx += 1
    
    #Get node parameters
    node = communities[c][idx]
    node_audio = audioFromNode(node, g)
    display(Audio(node_audio, rate=22050))

## spectral clustering

<div style="background-color:red;color:white; font-size:30px;padding:10px">
    Is it usefull ?
    

</div>

## Load Graph

In [ ]:
graph_path = "data\\Knn-G"
nb_hubs = 500
k = 35
graph_Name = "KnnG_Nhubs" + str(nb_hubs) +  "_K" + str(k) +".graphml"
graph_path = graph_path + "\\graphml_folder" + '\\' +  graph_Name 
print("Loadin + " graph_Name)
g = nx.read_graphml(graph_path)
g = g.to_undirected()
edge_type = "invDist" # "invDist" dist
affinity_type= "precomputed_nearest_neighbors" #Other solutions : "precomputed_nearest_neighbors" #"nearest_neighbors"
print("Done !")

In [ ]:
if edge_type == "invDist" :
    iter = 0
    eps = 1e-10
    for data in g.edges(data=True):
        data[2]["weight"] = 1/(data[2]["weight"]+eps)
        if iter<20:
            iter +=1;

In [ ]:
def largest_connected_subgraph(graph: nx.Graph) -> nx.Graph:
    """Return the largest connected component of a graph."""
    components = nx.connected_components(graph)
    component_size = []
    i = 0
    for c in components:
        component_size.append(len(c))
        i += 1
    largest_cc = max(nx.connected_components(graph), key=len)
    fig, ax = plt.subplots()
    ax.bar(np.linspace(0,len(component_size),len(component_size)),component_size)
    ax.set_yscale('log')
    ax.set_xlabel("Connected component")
    ax.set_ylabel("Number of nodes in the connected component")
    ax.set_title("Repartition of node in connected components")
    
    return graph.subgraph(largest_cc).copy()

G = largest_connected_subgraph(g)
print("Nb of nodes = ",len(G.nodes()))

## Process clustering

In [ ]:
def spectral_communities(
    graph: nx.Graph,
    n_clusters: int = 3,
    random_state: int = 0
):
    """
    Perform spectral clustering on a graph and return node communities.
    """
    adj = nx.to_scipy_sparse_array(graph)

    clustering = SpectralClustering(
        n_clusters=n_clusters,
        affinity=affinity_type,
        random_state=random_state,
        n_neighbors = k
    ).fit(adj)

    nodes = list(graph.nodes)
    communities = [
        {nodes[i] for i in np.where(clustering.labels_ == c)[0]}
        for c in range(n_clusters)
    ]
    return communities, clustering.labels_

In [ ]:
#Compute clustering
max_nb_clusters = 200
scores = []
clusters_list = []
cluster_data = {}

for nb_c in range(1,max_nb_clusters+1):
    print("Classification pour k = ",nb_c)
    communities, labels = spectral_communities(G, n_clusters=nb_c)
    score = nx.community.modularity(G,communities)
    #store data
    communities_list = [list(c) for c in communities]
    cluster_data[nb_c] = { "nodes" : communities_list,"score" :  score}
    clusters_list.append(communities)
    scores.append(score)
    print(scores[-1])

#save data in json file
data_file_name = "data\\Clustering\\" + graph_Name + "_" +edge_type + "_" + affinity_type + ".json"
cluster_data_JSON = json.dumps(cluster_data, indent=4)
with open(data_file_name, "w") as f:
    f.write(cluster_data_JSON)

In [ ]:
# Find the best clustering

nb_c_opti = np.argmax(scores) + 1
print("Nb optim de cluster :")
print(1 + nb_c_opti)
#plot the evolution of score

x = np.linspace(1,len(scores),len(scores))
plt.plot(x,scores)
plt.title("Score evolution with parameter : " + edge_type + " " + " " + affinity_type)
plt.xlabel("Number of clusters")
plt.ylabel("Score")
plt.savefig("score_eval_" + graph_Name + "_" +edge_type + "_" + affinity_type + ".svg",dpi=150)
plt.show()

## Plot saved spectral clusturing

### Recover data from the json file

In [ ]:
graph_path = "data\\Knn-G"
graph_Name = "KnnG_Nhubs285_K35"
graph_path = graph_path + '\\' +  graph_Name + ".graphml"
g = nx.read_graphml(graph_path)
g = g.to_undirected()
edge_type = "invDist" # "invDist" dist
affinity_type= "precomputed_nearest_neighbors" #"precomputed_nearest_neighbors" #"nearest_neighbors"

data_file_name = "data\\Clustering\\" + graph_Name + "_" +edge_type + "_" + affinity_type + ".json"
data = {}
with open(data_file_name, 'r') as file:
    data = json.load(file)

# recover list of scores
scores = np.array([data[k]['score'] for k in data.keys()])

### Find the best clustering

In [ ]:
nb_c_opti = np.argmax(scores) + 1
print("Nb optim de cluster :")
print(nb_c_opti)
#plot the evolution of score

x = np.linspace(1,len(scores),len(scores))
plt.plot(x,scores)
plt.title("Score evolution with parameter : " + edge_type + " " + " " + affinity_type)
plt.xlabel("Number of clusters")
plt.ylabel("Score")
plt.savefig("score_eval_" + graph_Name + "_" +edge_type + "_" + affinity_type + ".svg",dpi=150)
plt.show()

### Find the number of connexions between tow clusters

In [ ]:
nb_clusters = nb_c_opti
communities = data[str(nb_clusters)]['nodes']

w = nx.to_dict_of_dicts(g) 

clusters_connexions = np.zeros((nb_clusters,nb_clusters))
for c1 in range(len(communities)):
    for c2 in range(c1+1,len(communities)):
        # find nb connexion between c1 and c2
        nb_connections = 0
        for x1 in communities[c1].copy():
            for x2 in communities[c2].copy():
                if(x2 in w[x1]):
                    nb_connections += 1
            clusters_connexions[c1][c2] = nb_connections

In [ ]:
clusters_connexions  = clusters_connexions + clusters_connexions.T
plt.matshow(clusters_connexions/np.max(clusters_connexions))
plt.colorbar()
plt.show()

# Listen to some exemples
Listen to the most connected node in every cluster

In [ ]:
nb_clusters = nb_c_opti

communities = data[str(nb_clusters)]['nodes']

save_audio = False
audio_dir = "data\\audio\\" + graph_Name + "\\" +edge_type + "\\" + affinity_type


print("Nb of clusters : " + str(nb_clusters))

for c in range(len(communities)):
    # create a dict of degree
    d_a = {}
    for node in communities[c]:
        d_a[node] = g.degree()[node]
    #find correspoding node
    node_key = max(d_a, key=lambda k: d_a[k])
    idx = 0
    listNode = list(d_a.keys())
    while listNode[idx] != node_key:
        idx += 1
    #Get node parameters
    node_params = dict(g.nodes.data())[list(communities[c])[idx]]['features'].split(',')
    node_params = list(node_params)
    theta = [3]
    for x in node_params:
        theta.append(float(x))
    node_audio = rectangular_drum(theta, True,**constants)
    print("Cluster n°" + str(c) + " node : " + node_key)
    display(Audio(node_audio, rate=22050))
    node_audio = np.array(node_audio)
    if save_audio:
        write(audio_dir + "nb_cluster_" + str(nb_c_opti) + "\\Node_nb_" + str(c)+".wav", constants["sr"], node_audio)